# Setup e Imports

In [1]:
import os
import sys
import torch.nn as nn
import torch.optim as optim
import pandas as pd

# Descobre onde o notebook está rodando
current_dir = os.getcwd()

# Se estiver rodando dentro da pasta 'experiments', adiciona a pasta raiz ao path
if current_dir.endswith('experiments'):
    sys.path.append(os.path.abspath(os.path.join(current_dir, '..')))
else:
    sys.path.append(current_dir)

from core.config import DEVICE, DATASET_FINAL
from core.data import get_target_splits, get_dataloaders
from core.models import get_vibnet_model
from core.engine import pre_train_source, train_target_fold

print(f"Usando dispositivo: {DEVICE}")

Usando dispositivo: cuda


# Pré-treinamento (Source)

In [2]:
# Roda apenas uma vez (ou pula se os pesos já existirem)
w_imagenet = pre_train_source(target_to_exclude="CWRU", start_with_imagenet=True)
w_scratch = pre_train_source(target_to_exclude="CWRU", start_with_imagenet=False)

Pesos encontrados: /home/vfrocha/newvibnetexperiments/weights/vibnet_source_no_CWRU_imagenet.pth. Pulando pré-treino.
Pesos encontrados: /home/vfrocha/newvibnetexperiments/weights/vibnet_source_no_CWRU_scratch.pth. Pulando pré-treino.


# Configuração do Target

In [4]:
# Lista as condições
pu_root = os.path.join(DATASET_FINAL, "CWRU_12k")
conditions = sorted([d for d in os.listdir(pu_root) if os.path.isdir(os.path.join(pu_root, d))])
strategies = ["Scratch", "ImageNet", "VibNet_from_Scratch", "VibNet_from_ImageNet"]

print(f"Condições encontradas: {conditions}")

Condições encontradas: ['Load_0HP', 'Load_1HP', 'Load_2HP', 'Load_3HP']


# Experimento

In [6]:
results = []

for test_cond in conditions:
    print(f"\n{'='*40}\nFold: Testando em {test_cond}\n{'='*40}")
    
    train_x, train_y, test_x, test_y, num_classes = get_target_splits("CWRU_12k", test_cond)
    dataloaders = get_dataloaders(train_x, train_y, test_x, test_y)

    for strat in strategies:
        print(f"-> Estratégia: {strat}")
        
        w_path = w_imagenet if "ImageNet" in strat else w_scratch
        model = get_vibnet_model(num_classes, strat, w_path)
        
        lr = 0.001 if strat == "Scratch" else 0.0001
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        
        metrics = train_target_fold(model, dataloaders, optimizer, criterion, epochs=8)
        print(f"   Bal Acc: {metrics['Bal Accuracy']:.4f} | F1: {metrics['Macro F1']:.4f}")
        
        results.append({"Condition": test_cond, "Strategy": strat, **metrics})


Fold: Testando em Load_0HP
-> Estratégia: Scratch
   Bal Acc: 0.8795 | F1: 0.8753
-> Estratégia: ImageNet
   Bal Acc: 0.9308 | F1: 0.9309
-> Estratégia: VibNet_from_Scratch
   Bal Acc: 0.9983 | F1: 0.9983
-> Estratégia: VibNet_from_ImageNet
   Bal Acc: 0.9970 | F1: 0.9970

Fold: Testando em Load_1HP
-> Estratégia: Scratch
   Bal Acc: 0.9970 | F1: 0.9970
-> Estratégia: ImageNet
   Bal Acc: 0.9862 | F1: 0.9858
-> Estratégia: VibNet_from_Scratch
   Bal Acc: 0.9997 | F1: 0.9997
-> Estratégia: VibNet_from_ImageNet
   Bal Acc: 1.0000 | F1: 1.0000

Fold: Testando em Load_2HP
-> Estratégia: Scratch
   Bal Acc: 1.0000 | F1: 1.0000
-> Estratégia: ImageNet
   Bal Acc: 0.9997 | F1: 0.9997
-> Estratégia: VibNet_from_Scratch
   Bal Acc: 1.0000 | F1: 1.0000
-> Estratégia: VibNet_from_ImageNet
   Bal Acc: 1.0000 | F1: 1.0000

Fold: Testando em Load_3HP
-> Estratégia: Scratch
   Bal Acc: 0.9677 | F1: 0.9681
-> Estratégia: ImageNet
   Bal Acc: 0.9697 | F1: 0.9703
-> Estratégia: VibNet_from_Scratch
   B

# Resultados

In [7]:
df = pd.DataFrame(results)

resumo = df.groupby("Strategy")[["Bal Accuracy", "Macro F1"]].agg(['mean', 'std'])
display(resumo)

# Salva para anexar aos manuscritos depois
df.to_csv("resultados_cwru_unificado.csv", index=False)

Bal Accuracy            Macro F1          
                             mean       std      mean       std
Strategy                                                       
ImageNet                 0.971592  0.029826  0.971662  0.029723
Scratch                  0.961056  0.056273  0.960103  0.058305
VibNet_from_ImageNet     0.999260  0.001480  0.999252  0.001495
VibNet_from_Scratch      0.999502  0.000792  0.999493  0.000805